# 02 – Partição, pré-processamento e baseline

Executa depois do notebook 01. Em modo demonstração, a tabela de métricas abaixo vem de dados sintéticos e **não deve ser reportada** como baseline do projeto.

In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
import pandas as pd
from src import config
from src.dados import carregar_bruto, limpar, agregar_mensal, validar_totais
from src.demo import gerar_registros_sinteticos, ROTULO_DEMO
pd.set_option("display.width", 140)
from src.variaveis import criar_variaveis
from src.particao import particionar_temporal
from src.preprocessamento import construir_preprocessador, resumo_do_fit
from src.baseline import avaliar_baselines

In [ ]:
nome_real = config.PROCESSADO / "base_mensal.csv"
nome_demo = config.PROCESSADO / "base_mensal_DEMO.csv"
MODO_DEMO = not nome_real.exists()
arquivo = nome_demo if MODO_DEMO else nome_real
if MODO_DEMO:
    print(ROTULO_DEMO)
mensal = pd.read_csv(arquivo, parse_dates=[config.COL_MES])
print("Base lida:", arquivo.name, mensal.shape)

## Variáveis sem vazamento temporal
Defasagens e média móvel usam só meses anteriores ao mês-alvo.

In [ ]:
feats, descartadas = criar_variaveis(mensal)
print("Linhas usadas:", len(feats), "| descartadas por falta de histórico ou alvo ausente:", descartadas)
feats.head()

## Partição temporal
Treino no passado; validação e teste em meses posteriores. O teste fica guardado.

In [ ]:
treino, valid, teste = particionar_temporal(feats)
for nome, d in [("treino", treino), ("validação", valid), ("teste", teste)]:
    print(f"{nome:10s} {d[config.COL_MES].min():%Y-%m} a {d[config.COL_MES].max():%Y-%m}  linhas={len(d)}")

## Pré-processador ajustado só no treino

In [ ]:
cols = config.NUMERICAS + config.CATEGORICAS
pre = construir_preprocessador().fit(treino[cols])
resumo = resumo_do_fit(pre)
print("Medianas aprendidas no treino:", resumo["medianas"])
print("Médias aprendidas no treino:", resumo["medias"])
print("Categorias (unidade):", resumo["categorias"]["unidade"][:5], "...")
formas = {n: pre.transform(d[cols]).shape for n, d in [("treino", treino), ("validação", valid), ("teste", teste)]}
print("Formas após transformar:", formas)
assert len({f[1] for f in formas.values()}) == 1, "Conjuntos com número de atributos diferente"

## Baselines no recorte de validação
Métricas: MAE, WAPE e viés (previsto − real).

In [ ]:
tabela = avaliar_baselines(treino, valid).round(3)
tabela